# Autonomous neuron steering on Qwen3.5 4B

Claude Code (your subscription, via `CLAUDE_CODE_OAUTH_TOKEN`) acts as the interpretability agent. It finds MLP neurons that give Qwen3.5 4B a personality, then a fresh blind Claude judge scores the result against a system prompt and lyra's hand-found steers. Capability is checked with MMLU and GSM8K.

**Settings:** Accelerator **GPU T4 x2**, Internet **on**, and Add-ons > Secrets > `CLAUDE_CODE_OAUTH_TOKEN` attached. Then use **Save Version > Save & Run All** so the run continues in the background (it takes a few hours). Outputs land in `/kaggle/working`.

In [ ]:
DTYPE = "auto"   # auto = fp32 when 2 GPUs are visible (T4 x2), else fp16. T4 has no bf16.
AGENT_MAX_TURNS = 400
FAST_KERNELS = False  # True installs flash-linear-attention for faster DeltaNet layers; leave off if it errors on T4

In [ ]:
!pip install -q -U transformers accelerate datasets fastapi uvicorn requests
if FAST_KERNELS:
    !pip install -q flash-linear-attention
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%writefile /kaggle/working/neuron_server.py
"""Neuron-steering server for Qwen3.5 4B on a Kaggle GPU.

Runs on 127.0.0.1:8000. Steer recipe (lyra): a neuron n in layer L gets
dose * median(||resid leaving L||, tokens 1:) / ||W_down[:, n]|| added to its
down_proj input at every position, so dose=1 writes one median residual norm.
"""
import os, re, json, random, threading
import torch
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer

MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen3.5-4B")
HERE = os.path.dirname(os.path.abspath(__file__))
PROMPTS = json.load(open(os.path.join(HERE, "prompts.json")))
LOCK = threading.Lock()
STEERS = []  # active steers, read by the hooks: [(layer, neuron, value, mode)]


def load_model(dtype):
    kw = dict(dtype=dtype, device_map="auto")
    try:
        from transformers import AutoModelForCausalLM
        return AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw)
    except Exception as e:
        print("CausalLM load failed, trying ImageTextToText:", e)
        from transformers import AutoModelForImageTextToText
        return AutoModelForImageTextToText.from_pretrained(MODEL_ID, **kw)


tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

_dt = os.environ.get("DTYPE", "fp16")
if _dt == "auto":  # fp32 needs both T4s (~18 GB); a single GPU gets fp16
    _dt = "fp32" if torch.cuda.device_count() >= 2 else "fp16"
DTYPE = {"fp16": torch.float16, "fp32": torch.float32, "bf16": torch.bfloat16}[_dt]
model = load_model(DTYPE).eval()

# Text decoder layers only (skip any vision tower).
DOWN, LAYER = {}, {}
for name, mod in model.named_modules():
    if "visual" in name or "vision" in name:
        continue
    m = re.search(r"layers\.(\d+)\.mlp\.down_proj$", name)
    if m:
        DOWN[int(m.group(1))] = mod
    m = re.search(r"layers\.(\d+)$", name)
    if m:
        LAYER[int(m.group(1))] = mod
N_LAYERS = len(DOWN)
N_NEURONS = DOWN[0].in_features
W_NORM = {L: DOWN[L].weight.float().norm(dim=0).cpu() for L in DOWN}  # ||W_down[:, n]||
print(f"{MODEL_ID}: {N_LAYERS} layers x {N_NEURONS} neurons, dtype {DTYPE}")


def chat(prompt, system_prompt=None):
    msgs = ([{"role": "system", "content": system_prompt}] if system_prompt else []) + [{"role": "user", "content": prompt}]
    return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False, enable_thinking=False)


def encode(texts):
    return tok(texts, return_tensors="pt", padding=True).to(model.device)


# ---- steering hooks ---------------------------------------------------------
def make_hook(L):
    def hook(mod, args):
        active = [s for s in STEERS if s[0] == L]
        if not active:
            return None
        x = args[0].clone()
        for _, n, v, mode in active:
            if mode == "set":
                x[..., n] = v
            else:
                x[..., n] += v
        return (x,) + tuple(args[1:])
    return hook


for L in DOWN:
    DOWN[L].register_forward_pre_hook(make_hook(L))


# ---- calibration: median residual norm leaving each layer ---------------------
@torch.no_grad()
def calibrate():
    norms = {L: [] for L in LAYER}
    def grab(L):
        def h(mod, args, out):
            hs = out[0] if isinstance(out, tuple) else out
            norms[L].append(hs[0, 1:].float().norm(dim=-1).cpu())
        return h
    hs = [LAYER[L].register_forward_hook(grab(L)) for L in LAYER]
    try:
        for p in PROMPTS["calibration"]:
            model(**encode([chat(p)]))
    finally:
        for h in hs:
            h.remove()
    return {L: torch.cat(v).median().item() for L, v in norms.items()}


MEDIAN = calibrate()
print("median resid norms:", {L: round(v, 1) for L, v in MEDIAN.items()})


def resolve(steers):
    out = []
    for s in steers or []:
        L, n, d = int(s["layer"]), int(s["neuron"]), float(s["dose"])
        out.append((L, n, d * MEDIAN[L] / W_NORM[L][n].item(), s.get("mode", "add")))
    return out


# ---- core ops -----------------------------------------------------------------
@torch.no_grad()
def generate(prompts, system_prompt=None, max_new_tokens=200, temperature=0.0, batch=8):
    outs = []
    for i in range(0, len(prompts), batch):
        enc = encode([chat(p, system_prompt) for p in prompts[i:i + batch]])
        kw = dict(do_sample=True, temperature=temperature, top_p=0.95) if temperature > 0 else dict(do_sample=False)
        g = model.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **kw)
        outs += tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs


_mmlu, _gsm = None, None


def load_evals():
    global _mmlu, _gsm
    if _mmlu is None:
        from datasets import load_dataset
        m = list(load_dataset("cais/mmlu", "all", split="test"))
        random.Random(0).shuffle(m)
        _mmlu = m[:200]
        g = list(load_dataset("openai/gsm8k", "main", split="test"))
        random.Random(0).shuffle(g)
        _gsm = g[:100]
    return _mmlu, _gsm


LETTERS = [tok.encode(c, add_special_tokens=False)[0] for c in "ABCD"]


@torch.no_grad()
def mmlu_acc(items, system_prompt=None, batch=8):
    right = 0
    for i in range(0, len(items), batch):
        chunk = items[i:i + batch]
        texts = [chat(q["question"] + "\n" + "\n".join(f"{c}. {a}" for c, a in zip("ABCD", q["choices"]))
                      + "\nAnswer with the letter only.", system_prompt) for q in chunk]
        logits = model(**encode(texts), logits_to_keep=1).logits[:, -1, LETTERS]  # last token only; full logits OOM
        right += sum(int(p == q["answer"]) for p, q in zip(logits.argmax(-1).tolist(), chunk))
    return right / len(items)


def last_number(s):
    nums = re.findall(r"-?\d[\d,]*\.?\d*", s)
    return nums[-1].replace(",", "").rstrip(".") if nums else None


def gsm_acc(items, system_prompt=None):
    outs = generate([q["question"] + "\nThink step by step, then give the final number on the last line." for q in items],
                    system_prompt, max_new_tokens=320, batch=8)
    gold = [q["answer"].split("####")[-1].strip().replace(",", "") for q in items]
    return sum(last_number(o) == g for o, g in zip(outs, gold)) / len(items)


# ---- API ------------------------------------------------------------------------
app = FastAPI()


class Steer(BaseModel):
    layer: int
    neuron: int
    dose: float
    mode: str = "add"


class GenReq(BaseModel):
    prompts: list[str]
    steers: list[Steer] = []
    system_prompt: str | None = None
    max_new_tokens: int = 200
    temperature: float = 0.0


class CapReq(BaseModel):
    steers: list[Steer] = []
    system_prompt: str | None = None
    full: bool = False


class ContrastReq(BaseModel):
    positive: list[str]
    negative: list[str]
    top_k: int = 40
    layers: list[int] | None = None


@app.get("/health")
def health():
    return {"ok": True}


@app.get("/info")
def info():
    return {"model": MODEL_ID, "n_layers": N_LAYERS, "n_neurons": N_NEURONS, "dtype": str(DTYPE), "gpus": torch.cuda.device_count(),
            "median_resid_norm": MEDIAN}


@app.post("/generate")
def api_generate(r: GenReq):
    global STEERS
    with LOCK:
        STEERS = resolve([s.model_dump() for s in r.steers])
        try:
            return {"outputs": generate(r.prompts, r.system_prompt, r.max_new_tokens, r.temperature)}
        finally:
            STEERS = []


@app.post("/capability")
def api_capability(r: CapReq):
    """fast: 100 MMLU (logits only, ~1 min). full: 200 MMLU + 100 GSM8K (several minutes)."""
    global STEERS
    mmlu, gsm = load_evals()
    with LOCK:
        STEERS = resolve([s.model_dump() for s in r.steers])
        try:
            res = {"mmlu": mmlu_acc(mmlu if r.full else mmlu[:100], r.system_prompt)}
            if r.full:
                res["gsm8k"] = gsm_acc(gsm, r.system_prompt)
            return res
        finally:
            STEERS = []


@app.post("/contrast")
def api_contrast(r: ContrastReq):
    """Neurons whose mean down_proj input differs most between positive and negative texts
    (texts are fed raw, not chat-formatted). Score = mean difference / pooled std."""
    layers = r.layers or list(DOWN)
    acts = {L: {"pos": [], "neg": []} for L in layers}
    side = {"v": "pos"}
    def grab(L):
        def h(mod, args):
            acts[L][side["v"]].append(args[0][0, 1:].float().mean(0).cpu())
        return h
    with LOCK, torch.no_grad():
        hs = [DOWN[L].register_forward_pre_hook(grab(L)) for L in layers]
        try:
            for side["v"], texts in (("pos", r.positive), ("neg", r.negative)):
                for t in texts:
                    model(**encode([t]))
        finally:
            for h in hs:
                h.remove()
    found = []
    for L in layers:
        p, n = torch.stack(acts[L]["pos"]), torch.stack(acts[L]["neg"])
        diff = p.mean(0) - n.mean(0)
        score = diff / (torch.cat([p, n]).std(0) + 1e-6)
        for idx in score.abs().topk(r.top_k).indices.tolist():
            found.append({"layer": L, "neuron": idx, "score": round(score[idx].item(), 3),
                          "pos_mean": round(p.mean(0)[idx].item(), 4), "neg_mean": round(n.mean(0)[idx].item(), 4)})
    found.sort(key=lambda f: -abs(f["score"]))
    return {"neurons": found[:r.top_k]}

In [ ]:
%%writefile /kaggle/working/prompts.json
{
 "calibration": [
  "What is the capital of Australia?",
  "Explain how a refrigerator works.",
  "Write a haiku about rain.",
  "Give me three tips for a job interview.",
  "What causes the seasons on Earth?",
  "Summarize the plot of Romeo and Juliet.",
  "How do I make a cup of pour-over coffee?",
  "What is a prime number?",
  "Translate 'good morning' into Spanish and French.",
  "Why is the sky blue?",
  "Write a short thank-you note to a coworker.",
  "What are the pros and cons of electric cars?",
  "Explain recursion to a beginner.",
  "List five healthy breakfast ideas.",
  "How does compound interest work?",
  "What is photosynthesis?",
  "Describe the water cycle.",
  "Write a two-sentence horror story.",
  "How do vaccines work?",
  "What is the difference between a virus and bacteria?",
  "Give me a packing list for a weekend trip.",
  "Explain what an API is.",
  "What happened during the French Revolution?",
  "How do I change a flat tire?",
  "Write a limerick about a cat.",
  "What is machine learning?",
  "Explain supply and demand.",
  "How many planets are in the solar system?",
  "What should I consider when adopting a dog?",
  "Describe a sunset in one paragraph.",
  "What is the Pythagorean theorem?",
  "How do airplanes stay in the air?",
  "Write a polite email declining a meeting.",
  "What are black holes?",
  "Give me a simple pasta recipe.",
  "Explain how the stock market works.",
  "What is the boiling point of water at sea level?",
  "How do I start learning the guitar?",
  "Explain DNA in simple terms.",
  "What is the tallest mountain in the world?",
  "Write a motivational quote.",
  "How does GPS work?",
  "What are the main causes of climate change?",
  "Describe the rules of chess briefly.",
  "How do I improve my sleep?",
  "What is inflation?",
  "Explain the difference between weather and climate.",
  "Write a short poem about the ocean.",
  "How does a bill become a law in the US?",
  "What is the speed of light?",
  "Give me tips for public speaking.",
  "What is a neural network?",
  "How do bees make honey?",
  "Write a product description for a water bottle.",
  "What is the Great Wall of China?",
  "How do I budget my monthly expenses?",
  "Explain what a black box test is.",
  "What are renewable energy sources?",
  "How do I write a good resume?",
  "What is the theory of evolution?",
  "Describe how to plant a tomato.",
  "What is a sonnet?",
  "How do magnets work?",
  "What are some good study habits?"
 ],
 "heldout": [
  "I just got a promotion at work!",
  "My cat knocked my coffee off the table again.",
  "Can you help me plan a birthday party for my friend?",
  "What's the best way to learn a new language?",
  "I'm thinking about running a marathon. Thoughts?",
  "Explain how a microwave works.",
  "My code finally compiled after three hours.",
  "What should I cook for dinner tonight?",
  "Is it worth buying a standing desk?",
  "I failed my driving test today.",
  "Tell me about the Roman Empire.",
  "How do I ask my boss for a raise?",
  "What's your opinion on pineapple on pizza?",
  "I'm moving to a new city next month.",
  "Recommend a book for a long flight.",
  "Why do people like reality TV?",
  "I spilled red wine on a white carpet. Help.",
  "What's the point of learning calculus?",
  "My friend didn't reply to my text for two days.",
  "How do I keep houseplants alive?",
  "Explain blockchain like I'm five.",
  "I'm nervous about my presentation tomorrow.",
  "What's a good name for a goldfish?",
  "Should I get up early or stay up late to be productive?",
  "My laptop battery dies in an hour.",
  "What do you think about Mondays?",
  "I won a small raffle prize today.",
  "How do I deal with a noisy neighbor?",
  "Give me a fun fact about octopuses.",
  "I'm learning to bake bread. Any tips?"
 ],
 "system_prompts": {
  "sarcastic": "You are a sarcastic assistant. Answer helpfully, but with dry, sarcastic wit.",
  "chill friend": "You are a laid-back, chill friend. Talk casually and warmly, like a relaxed buddy, while still being helpful."
 },
 "lyra_steers": {
  "sarcastic": [
   {
    "layer": 15,
    "neuron": 3920,
    "dose": 1.0
   }
  ],
  "chill friend": [
   {
    "layer": 11,
    "neuron": 6261,
    "dose": 0.4
   },
   {
    "layer": 17,
    "neuron": 7870,
    "dose": -0.4
   }
  ]
 }
}

In [ ]:
%%writefile /kaggle/working/AGENT.md
# Task: find personality neurons in Qwen3.5 4B

You are an interpretability researcher. A steering server for Qwen/Qwen3.5-4B (32 layers, 9,216 MLP neurons per layer) is running at http://127.0.0.1:8000. Use `curl` or short Python scripts against it. Do not load the model yourself, and do not edit the files in this directory other than the outputs listed below.

Goal: for each trait, **"sarcastic"** and **"chill friend"**, find a small set of neuron steers (1 to 6 neurons) that makes the model clearly show the trait on everyday prompts, while MMLU drops by at most 2 points versus the unsteered model and replies stay coherent.

## Endpoints

- `GET /info` returns layer count, neuron count, and median residual norms.
- `POST /contrast {"positive": [texts], "negative": [texts], "top_k": 40, "layers": optional}` returns the neurons whose activations best separate trait-heavy text from neutral text (raw text, not chat-formatted). This is your main discovery tool. Write your own contrast texts: 15 to 30 each, matched in topic and length, differing only in the trait.
- `POST /generate {"prompts": [...], "steers": [{"layer": L, "neuron": n, "dose": d, "mode": "add"}], "system_prompt": optional, "max_new_tokens": 200}` returns steered replies (greedy). The neuron gets value = dose x median residual norm / ||W_down[:, n]|| at every token. `"mode": "add"` (default) adds that value to the neuron's activation, so dose 1.0 writes one median residual norm along its output direction; `"mode": "set"` clamps the activation to that value instead. Useful doses are usually between -1.5 and 1.5. Negative doses suppress. Text usually breaks down past about 1.5x the residual norm.
- `POST /capability {"steers": [...], "full": false}` returns MMLU accuracy on 100 questions (about a minute). `"full": true` adds 200 MMLU plus 100 GSM8K and takes several minutes.

## Rules

1. **Harness check first.** Lyra's public recipe says L15 N3920 at dose +1 makes this model sarcastic, but last run it only looked upbeat, so the steering setup may not match hers. Check `GET /info` for the dtype. On 8 of your own everyday prompts, generate with no steer, then with `[{"layer":15,"neuron":3920,"dose":D,"mode":M}]` for M in add and set and D in 0.5, 1.0 and 1.5. Score each setting for sarcasm and coherence. Pick the mode that reproduces sarcasm best and use it for every steer after this. If neither reproduces it, say so plainly, use add, and continue with the hunt anyway. Write `harness.json`: `{"mode": "add" or "set", "dtype": "...", "reproduced": true or false, "notes": "one line"}`, and record the outputs and scores in log.md. Don't reuse that neuron as your answer unless your own search independently finds it.
2. Never use the `heldout` prompts in prompts.json for discovery or tuning. Write your own development prompts (about 10 per trait).
3. Budget per trait: at most 25 calls to `/capability` with `full:false` and at most 3 with `full:true`. Get the unsteered baseline once with `full:true` and reuse it.
4. Score your own dev outputs on trait strength (1 to 10) and coherence (1 to 10). A steer that produces gibberish or repetition fails, however strong the trait.
5. Work like a scientist: form a hypothesis about what a neuron does, test it, and keep notes on what didn't work.
6. Don't give up early. Last run stopped after 26 turns. Use at least 150 turns (tool calls) of real search before deciding a trait can't be done, and try several discovery strategies (different contrast texts, per-layer searches, pairs of neurons, both dose signs, both modes).
7. Every steer you put in steers.json must have been tested at exactly that layer, neuron, dose and mode, on your dev prompts and with `/capability`. No untested placeholders. If nothing works for a trait, write your best tested attempt and say so.
8. The final eval also runs controls on your steers: random neurons at the same layers and doses, and your steers with the sign flipped. A real finding should beat both.

## Outputs (write these in the current directory)

- `harness.json`: see rule 1.
- `steers.json`: `{"sarcastic": [{"layer":..,"neuron":..,"dose":..,"mode":..}, ...], "chill friend": [...]}`, with your single best tested steer set per trait.
- `log.md`: every steer tried, with your dev trait/coherence scores and its MMLU, plus a short summary of what you learned about which layers and neurons matter.

Stop when both traits have a final steer that meets the goal, or when you've used the budget. In that case, write your best attempt and say plainly in log.md that it fell short.

In [ ]:
%%writefile /kaggle/working/JUDGE.md
# Task: blind-score chatbot replies

`blind.jsonl` holds chatbot replies, one JSON object per line: `id`, `trait`, `prompt`, `reply`. They come from several different setups mixed together, and you are not told which is which. Don't look for or open any other file in this directory.

For every line, score:

- `trait`, from 1 to 10: how clearly the reply shows the named trait. 1 means not at all and 10 means unmistakable throughout. "sarcastic" means dry, ironic, mocking wit. "chill friend" means casual, relaxed, and warm like a buddy.
- `coherence`, from 1 to 10: whether the reply is fluent, on-topic, and actually helpful. Repetition, gibberish, or ignoring the question score low.

Judge each reply on its own and use the full range. Go through the file in chunks, for example with a short Python script that prints 20 lines at a time. Write `scores.jsonl` with one line per id: `{"id": "...", "trait": n, "coherence": n}`. Every id must be scored exactly once.

In [ ]:
%%writefile /kaggle/working/final_eval.py
"""Final comparison, run after the agent. Generates replies on the held-out prompts for each setting per trait,
measures capability, and writes a blind pack for a fresh judge. A setting that fails is logged to eval_errors.json
and skipped, so one bad setting can't sink the run.

Controls per agent steer set: random_neuron (same layers, doses and mode, random neurons; 3 seeds, each on a third
of the held-out prompts, MMLU averaged) and sign_flip (every dose negated). Controls use the 100-question MMLU."""
import json, os, random, traceback, requests

API = "http://127.0.0.1:8000"
P = json.load(open("prompts.json"))
agent = json.load(open("steers.json"))
harness = json.load(open("harness.json")) if os.path.exists("harness.json") else {}
MODE = harness.get("mode", "add")
N_NEURONS = requests.get(API + "/info").json()["n_neurons"]
SEEDS = [0, 1, 2]


def post(path, body):
    r = requests.post(API + path, json=body, timeout=7200)
    r.raise_for_status()
    return r.json()


def with_mode(steers):
    return [{**s, "mode": s.get("mode", MODE)} for s in steers]


def random_like(steers, seed):
    rng = random.Random(seed)
    return [{**s, "neuron": rng.randrange(N_NEURONS)} for s in steers]


def flip(steers):
    return [{**s, "dose": -s["dose"]} for s in steers]


HO = P["heldout"]
third = len(HO) // len(SEEDS)
settings = {}  # key -> list of (steers, system_prompt, prompts, full_capability)
for trait in ["sarcastic", "chill friend"]:
    sp = P["system_prompts"][trait]
    mine = with_mode(agent.get(trait, []))
    s = {"base": [([], None, HO, True)],
         "system_prompt": [([], sp, HO, True)],
         "lyra_steer": [(with_mode(P["lyra_steers"][trait]), None, HO, True)]}
    if mine:
        s["agent_steer"] = [(mine, None, HO, True)]
        s["agent_steer+system_prompt"] = [(mine, sp, HO, True)]
        s["random_neuron"] = [(random_like(mine, sd), None, HO[i * third:(i + 1) * third], False) for i, sd in enumerate(SEEDS)]
        s["sign_flip"] = [(flip(mine), None, HO, False)]
    settings[trait] = s

capability, items, errors, used = {}, [], {}, {}
base_cap = {**post("/capability", {"steers": [], "full": True}), "n_mmlu": 200}
base_fast = {**post("/capability", {"steers": [], "full": False}), "n_mmlu": 100}
capability["unsteered|base (100q)"] = base_fast
for trait, conds in settings.items():
    for cond, runs in conds.items():
        key = f"{trait}|{cond}"
        print("running", key, flush=True)
        try:
            caps, outs = [], []
            for steers, sp, prompts, full in runs:
                if cond == "base":
                    caps.append(base_cap)
                else:
                    caps.append({**post("/capability", {"steers": steers, "system_prompt": sp, "full": full}),
                                 "n_mmlu": 200 if full else 100})
                outs += [{"key": key, "trait": trait, "prompt": p, "reply": o}
                         for p, o in zip(prompts, post("/generate", {"prompts": prompts, "steers": steers, "system_prompt": sp})["outputs"])]
            capability[key] = {k: sum(c[k] for c in caps) / len(caps) for k in caps[0]}
            used[key] = [r[0] for r in runs]
            items += outs
        except Exception as e:
            errors[key] = f"{type(e).__name__}: {e}\n" + traceback.format_exc()[-1500:]
            print("FAILED", key, errors[key][:300], flush=True)

random.Random(1).shuffle(items)
for i, it in enumerate(items):
    it["id"] = f"r{i:04d}"
json.dump({it["id"]: it["key"] for it in items}, open("blind_key.json", "w"), indent=1)
with open("blind.jsonl", "w") as f:
    for it in items:
        f.write(json.dumps({"id": it["id"], "trait": it["trait"], "prompt": it["prompt"], "reply": it["reply"]}) + "\n")
json.dump(capability, open("capability.json", "w"), indent=1)
json.dump(used, open("eval_steers.json", "w"), indent=1)
json.dump(errors, open("eval_errors.json", "w"), indent=1)
print(f"wrote {len(items)} blind items; {len(errors)} settings failed")

In [ ]:
%%writefile /kaggle/working/make_table.py
"""Join blind scores with the key and capability numbers into results.md. Missing scores, settings or files show as n/a."""
import json, os
from collections import defaultdict

load = lambda p, d: json.load(open(p)) if os.path.exists(p) else d
key = load("blind_key.json", {})
cap = load("capability.json", {})
errors = load("eval_errors.json", {})
harness = load("harness.json", {})
scores = defaultdict(lambda: {"trait": [], "coherence": []})
if os.path.exists("scores.jsonl"):
    for line in open("scores.jsonl"):
        try:
            s = json.loads(line)
            k = key[s["id"]]
            scores[k]["trait"].append(float(s["trait"]))
            scores[k]["coherence"].append(float(s["coherence"]))
        except Exception:
            continue

fmt = lambda v: f"{sum(v) / len(v):.1f}" if v else "n/a"
pct = lambda c, k: f"{c[k] * 100:.1f}" if isinstance(c.get(k), (int, float)) else "n/a"
keys = list(dict.fromkeys(list(cap) + sorted(set(key.values())) + list(errors)))
rows = [f"Harness: mode={harness.get('mode', 'add')}, dtype={harness.get('dtype', '?')}, "
        f"lyra sarcasm reproduced={harness.get('reproduced', '?')}", "",
        "| Trait | Setting | Trait (1-10) | Coherence (1-10) | MMLU | MMLU n | GSM8K | replies |",
        "|---|---|---|---|---|---|---|---|"]
for k in keys:
    trait, cond = k.split("|", 1)
    s, c = scores[k], cap.get(k, {})
    rows.append(f"| {trait} | {cond}{' (FAILED)' if k in errors else ''} | {fmt(s['trait'])} | {fmt(s['coherence'])} | "
                f"{pct(c, 'mmlu')} | {int(c['n_mmlu']) if 'n_mmlu' in c else 'n/a'} | {pct(c, 'gsm8k')} | {len(s['trait'])} |")
if errors:
    rows += ["", "Failed settings:"] + [f"- {k}: {v.splitlines()[0]}" for k, v in errors.items()]
open("results.md", "w").write("\n".join(rows) + "\n")
print("\n".join(rows))

In [ ]:
!curl -fsSL https://claude.ai/install.sh | bash
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
# Token: a Kaggle secret if attached, otherwise claude_token.txt in an attached private dataset.
import glob
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["CLAUDE_CODE_OAUTH_TOKEN"] = UserSecretsClient().get_secret("CLAUDE_CODE_OAUTH_TOKEN")
except Exception:
    os.environ["CLAUDE_CODE_OAUTH_TOKEN"] = open(glob.glob("/kaggle/input/**/claude_token.txt", recursive=True)[0]).read().strip()
!claude --version

In [ ]:
# Start the steering server in the background; first run downloads ~9 GB of weights.
import subprocess, time, requests
os.chdir("/kaggle/working")
server = subprocess.Popen(["python", "-m", "uvicorn", "neuron_server:app", "--host", "127.0.0.1", "--port", "8000"],
                          stdout=open("server.log", "w"), stderr=subprocess.STDOUT, env={**os.environ, "DTYPE": DTYPE})
for _ in range(180):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).ok: break
    except Exception: pass
    if server.poll() is not None: break
    time.sleep(10)
print(open("server.log").read()[-3000:])
print(requests.get("http://127.0.0.1:8000/info").json())

In [ ]:
# Sanity check: unsteered vs lyra's sarcastic neuron. Both should be fluent English.
q = ["My cat knocked my coffee off the table again.", "What should I cook tonight?"]
for steers in ([], [{"layer": 15, "neuron": 3920, "dose": 1.0}]):
    out = requests.post("http://127.0.0.1:8000/generate", json={"prompts": q, "steers": steers, "max_new_tokens": 80}).json()
    print(steers, "\n", "\n---\n".join(out["outputs"]), "\n=====")

In [ ]:
import json
def run_claude(prompt, cwd, max_turns):
    """Headless Claude Code; prints a compact live trace and saves the raw stream."""
    cmd = ["claude", "-p", prompt, "--output-format", "stream-json", "--verbose",
           "--allowedTools", "Bash Read Write Edit Glob Grep", "--max-turns", str(max_turns)]
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with open(os.path.join(cwd, "claude_stream.jsonl"), "a") as raw:
        for line in p.stdout:
            raw.write(line)
            try: ev = json.loads(line)
            except Exception: print(line, end=""); continue
            if ev.get("type") == "assistant":
                for c in ev["message"].get("content", []):
                    if c.get("type") == "text": print("CLAUDE:", c["text"][:800])
                    elif c.get("type") == "tool_use": print("TOOL:", json.dumps(c.get("input"))[:400])
            elif ev.get("type") == "result":
                print("RESULT:", ev.get("subtype"), "turns:", ev.get("num_turns"))
    return p.wait()

run_claude(open("/kaggle/working/AGENT.md").read(), "/kaggle/working", AGENT_MAX_TURNS)
print(open("/kaggle/working/steers.json").read())

In [ ]:
!cd /kaggle/working && python final_eval.py

In [ ]:
# Blind judge: a fresh Claude Code session in a separate folder that cannot see the key.
os.makedirs("/kaggle/working/judge", exist_ok=True)
!cp /kaggle/working/blind.jsonl /kaggle/working/judge/
run_claude(open("/kaggle/working/JUDGE.md").read(), "/kaggle/working/judge", 300)
!cp /kaggle/working/judge/scores.jsonl /kaggle/working/; cd /kaggle/working && python make_table.py

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("/kaggle/working/results.md").read()))
print(open("/kaggle/working/log.md").read()[-4000:])